# 06. ALOHA 2 양팔 로봇(Bimanual Manipulation) 시뮬레이션 및 벤치마크 태스크 실습

![ALOHA Sim Banner](../assets/imgs/aloha/banner.png)

이 노트북에서는 Google DeepMind의 **ALOHA 2 양팔 로봇 시뮬레이션 패키지(`aloha_sim`)**를 활용하여, 단일 로봇 팔을 넘어 **양팔 협동 매니퓰레이터(Bimanual Manipulator)**의 구조, 멀티 카메라 비전 관측, 양팔 미분 역기구학(Bimanual IK), 그리고 `dm_control.composer` 기반의 **20종 일상 물체 조작 태스크**를 실습합니다.

---
### 학습 목표
1. **ALOHA 2 하드웨어 및 MJCF 모델 구조 이해**: 양팔(ViperX 300s 6-DoF × 2)과 평행 그리퍼(Equality 제약 조건 연동), 4대의 카메라(`overhead_cam`, `worms_eye_cam`, `wrist_cam_left`, `wrist_cam_right`) 이해하기
2. **양팔 미분 역기구학 (Bimanual Differential IK)**: 왼손(`left/gripper`)과 오른손(`right/gripper`)의 6자유도 목표 포즈를 동시에 제어하기
3. **`aloha_sim` 태스크 스위트(`task_suite`) 활용**: `dm_control.composer`로 구성된 20종의 조작 환경(`HandOverBanana`, `DrawerOpen`, `LaptopClose` 등)을 불러오고 관측값(`observation`), 행동 공간(`action_spec`), 보상(`reward`) 다루기
4. **Windows 인터랙티브 뷰어 실행**: 노트북에서 바로 ALOHA 2 작업 환경을 3D 인터랙티브 뷰어로 띄워 물체 및 로봇과 실시간 상호작용하기

In [ ]:
import time
import mujoco
import mujoco.viewer
import mediapy as media
import matplotlib.pyplot as plt
import numpy as np

from tasks import task_suite
from tasks.base import aloha2_task

# Windows 환경 초기화 (내장 FFmpeg 자동 연결 및 한글 폰트 설정)
import sys
from pathlib import Path
for _p in [Path.cwd() / "src", Path.cwd().parent / "src"]:
    if _p.exists() and str(_p.resolve()) not in sys.path:
        sys.path.insert(0, str(_p.resolve()))
from utils.notebook_setup import init_notebook
init_notebook()

## 1. 순수 MuJoCo API로 ALOHA 2 PBR 모델 탐색하기

먼저 `assets/robots/aloha/scene_pbr.xml` 파일을 순수 `mujoco.MjModel`로 불러와 양팔 로봇의 관절(`joint`), 액추에이터(`actuator`), 그리퍼 사이트(`site`), 그리고 카메라(`camera`) 구성을 살펴보겠습니다.

### ALOHA 2의 자유도(`nq = 16`) vs 제어 입력(`nu = 14`)
- **왼팔(Left Arm)**: 6개 회전 관절(`waist`, `shoulder`, `elbow`, `forearm_roll`, `wrist_angle`, `wrist_rotate`) + 2개 직동 손가락 관절(`left_finger`, `right_finger`) = **8 DoF**
- **오른팔(Right Arm)**: 6개 회전 관절 + 2개 직동 손가락 관절 = **8 DoF**
- 각 그리퍼의 양쪽 손가락(`left_finger`, `right_finger`)은 MJCF의 `<equality>` 태그를 통해 대칭으로 함께 움직이도록 구속되어 있으므로, 실제 제어 액추에이터(`data.ctrl`)는 **왼팔 7개(관절 6 + 그리퍼 1) + 오른팔 7개(관절 6 + 그리퍼 1) = 총 14개**입니다!

In [ ]:
# ALOHA 2 PBR 씬 XML 불러오기
aloha_xml_path = "assets/robots/aloha/scene_pbr.xml"
model = mujoco.MjModel.from_xml_path(aloha_xml_path)
data = mujoco.MjData(model)

print(f"총 일반화 위치 차원 (nq): {model.nq}")
print(f"총 자유도 차원 (nv):      {model.nv}")
print(f"총 액추에이터 수 (nu):    {model.nu}")
print("\n[관절(Joint) 목록]:")
for i in range(model.njnt):
    print(f"  {i:2d}: {model.joint(i).name:22s} (qpos 주소: {model.jnt_qposadr[i]:2d})")

print("\n[카메라(Camera) 목록]:")
for i in range(model.ncam):
    print(f"  {i:2d}: {model.camera(i).name}")

### 4개의 내장 카메라 멀티뷰(Multi-View) 렌더링
ALOHA 2 로봇 셀에는 모방 학습(Imitation Learning) 및 시각-운동 정책(Visuomotor Policy)을 위해 4대의 카메라가 장착되어 있습니다:
1. **`overhead_cam`**: 테이블 전체를 위에서 내려다보는 상단 고정 카메라
2. **`worms_eye_cam`**: 테이블 앞쪽 아래에서 위를 올려다보는 하단 고정 카메라
3. **`wrist_cam_left`**: 왼쪽 그리퍼 손목에 부착된 1인칭(Eye-in-Hand) 카메라
4. **`wrist_cam_right`**: 오른쪽 그리퍼 손목에 부착된 1인칭(Eye-in-Hand) 카메라

로봇을 기본 홈 자세(`HOME_QPOS`)로 설정하고 4개 카메라 시점을 동시에 렌더링해 보겠습니다:

In [ ]:
# 양팔을 기본 홈 자세(HOME_QPOS)로 초기화
data.qpos[:8] = aloha2_task.HOME_QPOS
data.qpos[8:16] = aloha2_task.HOME_QPOS
data.ctrl[:7] = aloha2_task.HOME_CTRL
data.ctrl[7:14] = aloha2_task.HOME_CTRL
mujoco.mj_forward(model, data)

camera_names = ["overhead_cam", "worms_eye_cam", "wrist_cam_left", "wrist_cam_right"]
renderer = mujoco.Renderer(model, height=360, width=640)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, cam_name in zip(axes.flatten(), camera_names):
    renderer.update_scene(data, camera=cam_name)
    img = renderer.render()
    ax.imshow(img)
    ax.set_title(f"Camera: {cam_name}")
    ax.axis("off")

plt.tight_layout()
plt.show()
renderer.close()

---
## 2. 양팔 미분 역기구학 (Bimanual Differential IK) 제어 실습

`02_robot_arms.ipynb`에서 배운 단일 팔 미분 역기구학(Differential IK)을 **양팔(Bimanual) 시스템**으로 확장해 보겠습니다.
왼손 그리퍼 중심 사이트(`left/gripper`)와 오른손 그리퍼 중심 사이트(`right/gripper`)에 대해 각각 자코비안 $J_L, J_R$을 구하고, 왼팔 6개 관절(`qpos[0:6]`)과 오른팔 6개 관절(`qpos[8:14]`)을 독립적으로 업데이트하면서 그리퍼 개폐(`ctrl[6]`, `ctrl[13]`)를 동시에 제어합니다:

In [ ]:
# 양팔 그리퍼 사이트 ID 조회
left_site_id = model.site("left/gripper").id
right_site_id = model.site("right/gripper").id

# 홈 자세로 초기화 후 초기 그리퍼 위치/자세 저장
data.qpos[:8] = aloha2_task.HOME_QPOS
data.qpos[8:16] = aloha2_task.HOME_QPOS
data.ctrl[:7] = aloha2_task.HOME_CTRL
data.ctrl[7:14] = aloha2_task.HOME_CTRL
mujoco.mj_forward(model, data)

left_init_pos = data.site(left_site_id).xpos.copy()
right_init_pos = data.site(right_site_id).xpos.copy()
left_init_quat = np.zeros(4)
right_init_quat = np.zeros(4)
mujoco.mju_mat2Quat(left_init_quat, data.site(left_site_id).xmat)
mujoco.mju_mat2Quat(right_init_quat, data.site(right_site_id).xmat)

def step_arm_ik(model, data, site_id, target_pos, target_quat, dof_slice, step_gain=0.5):
    """특정 팔(왼팔 또는 오른팔)의 6개 관절에 대해 미분 역기구학 변위(dq)를 계산합니다."""
    jac = np.zeros((6, model.nv))
    error = np.zeros(6)
    site_quat = np.zeros(4)
    target_quat_conj = np.zeros(4)
    error_quat = np.zeros(4)

    # 위치 오차
    error[:3] = data.site(site_id).xpos - target_pos
    # 자세 오차
    mujoco.mju_negQuat(target_quat_conj, target_quat)
    mujoco.mju_mat2Quat(site_quat, data.site(site_id).xmat)
    mujoco.mju_mulQuat(error_quat, site_quat, target_quat_conj)
    mujoco.mju_quat2Vel(error[3:], error_quat, 1.0)

    # 전체 자코비안 계산 후 해당 팔의 6개 관절 열(dof_slice)만 추출
    mujoco.mj_jacSite(model, data, jac[:3], jac[3:], site_id)
    jac_arm = jac[:, dof_slice]

    # 감쇠 최소제곱(Damped Least Squares) 역기구학 해 계산
    lmbda = 1e-2
    dq = -step_gain * jac_arm.T @ np.linalg.inv(jac_arm @ jac_arm.T + lmbda * np.eye(6)) @ error
    return dq

# 4초간 양손이 중앙으로 마주 다가오며 원을 그리고 그리퍼를 여닫는 모션 시뮬레이션
duration = 4.0
framerate = 30
frames = []
renderer = mujoco.Renderer(model, height=360, width=640)

while data.time < duration:
    t = data.time
    # 왼손과 오른손의 대칭 3D 목표 위치 생성
    offset_l = np.array([0.08 * (1 - np.cos(2 * np.pi * 0.5 * t)),  0.08 * np.sin(2 * np.pi * 0.5 * t), 0.05 * np.sin(2 * np.pi * 0.5 * t)])
    offset_r = np.array([-0.08 * (1 - np.cos(2 * np.pi * 0.5 * t)), -0.08 * np.sin(2 * np.pi * 0.5 * t), 0.05 * np.sin(2 * np.pi * 0.5 * t)])

    dq_left = step_arm_ik(model, data, left_site_id, left_init_pos + offset_l, left_init_quat, slice(0, 6))
    dq_right = step_arm_ik(model, data, right_site_id, right_init_pos + offset_r, right_init_quat, slice(8, 14))

    # 양팔 관절 위치 명령 업데이트
    data.ctrl[0:6] = data.qpos[0:6] + dq_left
    data.ctrl[7:13] = data.qpos[8:14] + dq_right

    # 그리퍼 주기적 개폐 (0.002m = 닫힘, 0.037m = 열림)
    grip_val = 0.0195 + 0.0175 * np.sin(2 * np.pi * 1.0 * t)
    data.ctrl[6] = grip_val
    data.ctrl[13] = grip_val

    mujoco.mj_step(model, data)

    if len(frames) < data.time * framerate:
        renderer.update_scene(data, camera="teleoperator_pov")
        frames.append(renderer.render())

renderer.close()
media.show_video(frames, fps=framerate)

---
## 3. `aloha_sim` 20종 양팔 조작 벤치마크 태스크(`task_suite`) 탐색

`tasks.task_suite`는 `dm_control.composer`를 사용하여 ALOHA 2 로봇 작업대 위에 다양한 일상 물체(바나나, 그릇, 머그컵, 서랍장, 노트북, 드라이버, 가위, 마커펜, 수건 등)를 배치하고 작업 성공 여부를 자동으로 판정해 주는 **20종의 표준 태스크**를 제공합니다.

등록된 모든 태스크 목록을 확인해 보겠습니다:

In [ ]:
print(f"총 등록된 ALOHA Sim 태스크 수: {len(task_suite.TASK_FACTORIES)}종\n")
for idx, name in enumerate(task_suite.TASK_FACTORIES.keys(), 1):
    print(f"  {idx:2d}. {name}")

### 대표 태스크 환경 생성 및 관측/행동 명세 확인 (`HandOverBanana`)
바나나를 한 손으로 집어 반대편 손으로 건네준 뒤 파란 그릇 안에 넣는 **`HandOverBanana`** 환경을 생성하고, 환경이 제공하는 언어 지시문(`get_instruction()`), 관측 공간(`observation_spec()`), 행동 공간(`action_spec()`)을 확인해 보겠습니다:

In [ ]:
# HandOverBanana 태스크 환경 생성
env = task_suite.create_task_env(
    task_name="HandOverBanana",
    time_limit=10.0,
    random_state=42,
    camera_resolution=(360, 640),
)
timestep = env.reset()

print("작업 지시문 (Instruction):", env.task.get_instruction())
print("행동 공간 (Action Spec):  ", env.action_spec())
print("\n[관측값 (Observation Keys & Shapes)]:")
for k, v in timestep.observation.items():
    print(f"  - {k:24s}: shape={v.shape}, dtype={v.dtype}")

`timestep.observation` 안에 이미 4개 카메라의 RGB 이미지 배열이 포함되어 있으므로 별도의 렌더러를 만들 필요 없이 바로 시각화할 수 있습니다:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, cam_name in zip(axes.flatten(), task_suite.DEFAULT_CAMERAS):
    ax.imshow(timestep.observation[cam_name])
    ax.set_title(f"HandOverBanana - {cam_name}")
    ax.axis("off")
plt.tight_layout()
plt.show()
env.close()

### 다양한 조작 태스크 6종 한눈에 비교하기
서랍 열기(`DrawerOpen`), 노트북 닫기(`LaptopClose`), 식기 건조대에 그릇 올리기(`BowlOnRack`), 마커펜 뚜껑 열기(`MarkerRemoveLid`), 공구함에 드라이버 넣기(`ToolsPlaceScrewdriverInLeftCompartment`), 알파벳 블록 배열하기(`BlocksSpelling`) 등 6가지 태스크의 초기 화면을 비교해 보겠습니다:

In [ ]:
sample_tasks = [
    "DrawerOpen",
    "LaptopClose",
    "BowlOnRack",
    "MarkerRemoveLid",
    "ToolsPlaceScrewdriverInLeftCompartment",
    "BlocksSpelling",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, t_name in zip(axes.flatten(), sample_tasks):
    temp_env = task_suite.create_task_env(
        task_name=t_name,
        time_limit=5.0,
        random_state=0,
        cameras=("overhead_cam",),
        camera_resolution=(360, 640),
    )
    ts = temp_env.reset()
    instr = temp_env.task.get_instruction()
    ax.imshow(ts.observation["overhead_cam"])
    ax.set_title(f"{t_name}\n\"{instr}\"", fontsize=10)
    ax.axis("off")
    temp_env.close()

plt.tight_layout()
plt.show()

---
## 4. 태스크 환경에서의 에피소드 실행 및 멀티뷰 비디오 녹화 (`LaptopClose`)

이번에는 테이블 위에 열려 있는 노트북을 닫는 **`LaptopClose`** 태스크 환경을 생성하고, 오른팔을 노트북 상판 뒤쪽으로 이동시켜 앞으로 밀어 닫는 간단한 궤적 제어 루프를 실행하며 상단 카메라(`overhead_cam`)와 오른손 손목 카메라(`wrist_cam_right`)를 나란히 이어 붙인 비디오를 만들어 보겠습니다:

In [ ]:
env = task_suite.create_task_env(
    task_name="LaptopClose",
    time_limit=4.0,
    random_state=42,
    cameras=("overhead_cam", "wrist_cam_right"),
    camera_resolution=(360, 480),
)
timestep = env.reset()

# 초기 행동(양팔 홈 포즈 + 그리퍼 열림)
home_action = np.concatenate([
    aloha2_task.HOME_CTRL[:6], [aloha2_task.FOLLOWER_GRIPPER_OPEN],
    aloha2_task.HOME_CTRL[:6], [aloha2_task.FOLLOWER_GRIPPER_OPEN],
])

frames = []
for step_idx in range(80):
    t = step_idx / 80.0
    action = home_action.copy()
    # 오른팔(인덱스 7~12)을 중앙 노트북 방향으로 부드럽게 스윙
    action[7] = 0.25 * np.sin(np.pi * t)        # right/waist
    action[8] = -0.96 + 0.35 * np.sin(np.pi * t) # right/shoulder
    action[9] = 1.16 - 0.25 * np.sin(np.pi * t)  # right/elbow

    timestep = env.step(action)
    if step_idx % 2 == 0:
        # 상단 카메라와 오른쪽 손목 카메라 이미지를 좌우로 결합
        combined_frame = np.concatenate([
            timestep.observation["overhead_cam"],
            timestep.observation["wrist_cam_right"]
        ], axis=1)
        frames.append(combined_frame)

env.close()
print("좌측: 상단 카메라(overhead_cam) | 우측: 오른손 손목 카메라(wrist_cam_right)")
media.show_video(frames, fps=25)

---
## 5. Windows 인터랙티브 3D 뷰어로 ALOHA 2 태스크 직접 체험하기

아래 셀에서 원하는 `selected_task` 이름을 지정하고 실행하면 **별도의 MuJoCo 3D 인터랙티브 뷰어 창**이 열립니다!

### 인터랙티브 뷰어 조작법
- **카메라 회전 / 이동 / 줌**: 마우스 왼쪽 드래그 / 오른쪽 드래그 / 휠 스크롤
- **물체 또는 로봇에 힘 가하기**: 바나나, 서랍, 노트북 덮개 등을 **마우스 더블클릭**하여 선택한 뒤, **`Ctrl` + 마우스 오른쪽 드래그**로 직접 당기거나 밀어보세요!
  - 예를 들어 `DrawerOpen`에서 서랍 손잡이를 더블클릭 후 `Ctrl + 우클릭 드래그`로 열거나, `HandOverBanana`에서 바나나를 파란 그릇 안으로 옮기면 환경의 성공 감지기(`get_reward() == 1.0`)가 작동을 감지하고 콘솔에 성공 메시지를 출력합니다!
- **종료**: 뷰어 창을 닫거나 `ESC` 키를 누르면 셀 실행이 종료됩니다.

In [ ]:
# 체험하고 싶은 태스크 이름을 선택하세요 (예: "HandOverBanana", "DrawerOpen", "LaptopClose", "BowlOnRack")
selected_task = "HandOverBanana"

env = task_suite.create_task_env(
    task_name=selected_task,
    time_limit=float("inf"),
    random_state=0,
    cameras=("overhead_cam",),
    camera_resolution=(360, 640),
)
timestep = env.reset()
print(f"[{selected_task}] 지시문: \"{env.task.get_instruction()}\"")
print("인터랙티브 뷰어를 시작합니다. 물체를 더블클릭 후 Ctrl + 마우스 드래그로 움직여 보세요!")

viewer_model = env.physics.model.ptr
viewer_data = env.physics.data.ptr

home_action = np.concatenate([
    aloha2_task.HOME_CTRL[:6], [aloha2_task.FOLLOWER_GRIPPER_OPEN],
    aloha2_task.HOME_CTRL[:6], [aloha2_task.FOLLOWER_GRIPPER_OPEN],
])

with mujoco.viewer.launch_passive(viewer_model, viewer_data) as viewer_handle:
    viewer_handle.sync()
    while viewer_handle.is_running():
        step_start = time.time()
        timestep = env.step(home_action)

        # 마우스 외력(Perturbation) 적용
        with viewer_handle.lock():
            if viewer_handle.perturb.active:
                mujoco.mjv_applyPerturbForce(
                    viewer_model, viewer_data, viewer_handle.perturb
                )
            viewer_handle.sync()

        if timestep.reward >= 1.0:
            print("🎉 태스크 성공(Reward = 1.0)! 에피소드를 리셋합니다.")
            timestep = env.reset()

        elapsed = time.time() - step_start
        if elapsed < 0.02:
            time.sleep(0.02 - elapsed)

env.close()
print("인터랙티브 뷰어가 종료되었습니다.")